In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from datasets import load_dataset
from copy import deepcopy
import numpy as np
from scipy.stats import ttest_ind
import torch
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df = pd.read_csv('../QA_datasets_classified_qa_eval/output_csv/GaRAGe_UND_gpt4o_Ragas.csv')
ragas_col = df['ragas_AA_short'].tolist()
pre_ragas = load_dataset("json", 
                        data_files="../QA_datasets_classified_qa_eval/intermediate/BASELINE_GaRAGe_UND_qa_gpt_with_squad_scores.jsonl",
                        split="all")
assert len(pre_ragas) == len(ragas_col), "Fatal Error: length mismatch"

Generating train split: 603 examples [00:00, 25729.57 examples/s]


In [3]:
with_ragas = pre_ragas.add_column('ragas_AA_short', ragas_col)
with_ragas.to_json("./intermediate/GaRAGe_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 26.39ba/s]


2726693

## Rewriting with Gemini
GPT-4o rewriting, then GPT-4o QA later

In [4]:
from helper_functions_qr import modification_in_batch

In [5]:
client = OpenAI(
    api_key=os.environ.get('GOOGLE_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model = "gemini-2.5-flash"
input_file = "./intermediate/GaRAGe_UND_gpt4o_Ragas.jsonl"
output_file = "./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl"



In [6]:
# 清空输出文件（如果存在）
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Cleared existing output file: {output_file}")

# 处理所有样本（按批次）
question_modification = modification_in_batch(input_file, output_file, 'answer', client, model)

Total samples to process: 603
Batch size: 3


Processing batches:  89%|████████▊ | 178/201 [1:53:22<14:38, 38.18s/it]  

Error processing sample 536: Invalid \escape: line 3 column 43 (char 137)


Processing batches: 100%|██████████| 201/201 [2:07:27<00:00, 38.05s/it]


All batch processing completed! Total processed: 603 samples
Results saved to: ./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl


In [7]:
df_view = pd.DataFrame(question_modification)
df_view

,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA
0,What strategies can businesses employ to mitig...,What comprehensive strategies can U.S.-based b...,"[To mitigate ASC 842 compliance challenges, bu...",[- Implement lease accounting software. \n- C...,The query seeks strategies to address ASC 842 ...,0.153846,0,0.75
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Retinal Convolutional Neural...,[The Deep Retinal Convolutional Neural Network...,[- It improves accuracy by leveraging deep lea...,The query references 'Deep Retinal Convolution...,0.136546,0,0.50
2,"How does Zoe Law's ""Legends"" exhibition reflec...","Given that Zoë Law's ""Legends"" exhibition show...","[The ""Legends"" exhibition by Zoë Law reflects ...","['Legends' explores themes of identity, herita...",The query specifies a particular artist (Zoe L...,0.138614,0,0.25
3,How does the involvement of Tyco Ventures and ...,What is the impact of the $25 million investme...,[The involvement of Tyco Ventures and Integral...,[Tyco Ventures and Integral Capital Partners p...,The query identifies specific entities (Tyco V...,0.303797,0,0.50
4,How has the Drake-Kendrick Lamar feud influenc...,"Since its inception with Kendrick Lamar's ""Con...",[The feud between Drake and Kendrick Lamar has...,[- Increased focus on lyrical competition and ...,The query identifies a specific subject (the D...,0.158333,0,1.00
...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,Considering the proposed continuing resolution...,[Elon Musk's opposition led to the president-e...,[Elon Musk's opposition to Ukraine funding rep...,The query lacks critical specificity. It refer...,0.101695,0,0.00
599,What challenges does self-managed OpenSearch d...,What are the primary technical and operational...,[Self-managed OpenSearch deployments face seve...,[- Complex setup and configuration. \n- Ongoi...,The query asks about challenges associated wit...,0.137339,0,1.00
600,What are the implications of Cleveland-Cliffs ...,What is the context for Cleveland-Cliffs CEO's...,[Cleveland-Cliffs CEO's plan to make another o...,[- Potential prolonged negotiations and uncert...,The query specifies key elements: the subject ...,0.179775,0,0.00
601,How has the expansion of telehealth services u...,Considering the ongoing expansion of telehealt...,[The expansion of telehealth services under Me...,[- Improved access to healthcare for rural res...,The query specifies the subject matter (telehe...,0.293103,0,1.00


## Modified queries QA using GPT-4o

### Loading modified data

In [8]:
modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)


#modified_set = modified_set.remove_columns(["model_new_answer"])

#modified_set.to_json(
    #"./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl",
    #orient="records",
    #lines=True
#)

Generating train split: 603 examples [00:00, 48480.23 examples/s]


### Implementation

In [9]:
from helper_functions_qr import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [10]:
modified_results = batch_QA_with_progress(
    modified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_new_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_new_answer: 100%|██████████| 61/61 [19:59<00:00, 19.66s/it]


In [11]:
qa_modified = deepcopy(modified_set)
for key in modified_results:
    qa_modified = qa_modified.add_column(key, modified_results[key])

qa_modified.to_json("./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl", orient="records", lines=True)
df_qa_modified = pd.read_json("./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl", lines=True)
#df_qa_modified.to_csv('produced_files/modification_pilot_qa.csv')
df_qa_modified

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 39.36ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer
0,What strategies can businesses employ to mitig...,What comprehensive strategies can U.S.-based b...,"[To mitigate ASC 842 compliance challenges, bu...",[- Implement lease accounting software. \n- C...,The query seeks strategies to address ASC 842 ...,0.153846,0,0.75,[- Implement robust lease accounting software ...
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Retinal Convolutional Neural...,[The Deep Retinal Convolutional Neural Network...,[- It improves accuracy by leveraging deep lea...,The query references 'Deep Retinal Convolution...,0.136546,0,0.50,[- DCNN improves speech emotion recognition by...
2,"How does Zoe Law's ""Legends"" exhibition reflec...","Given that Zoë Law's ""Legends"" exhibition show...","[The ""Legends"" exhibition by Zoë Law reflects ...","['Legends' explores themes of identity, herita...",The query specifies a particular artist (Zoe L...,0.138614,0,0.25,[- It celebrates resilience by showcasing infl...
3,How does the involvement of Tyco Ventures and ...,What is the impact of the $25 million investme...,[The involvement of Tyco Ventures and Integral...,[Tyco Ventures and Integral Capital Partners p...,The query identifies specific entities (Tyco V...,0.303797,0,0.50,[- Strengthened financial position for Scion P...
4,How has the Drake-Kendrick Lamar feud influenc...,"Since its inception with Kendrick Lamar's ""Con...",[The feud between Drake and Kendrick Lamar has...,[- Increased focus on lyrical competition and ...,The query identifies a specific subject (the D...,0.158333,0,1.00,[- Elevated the importance of lyrical prowess ...
...,...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,Considering the proposed continuing resolution...,[Elon Musk's opposition led to the president-e...,[Elon Musk's opposition to Ukraine funding rep...,The query lacks critical specificity. It refer...,0.101695,0,0.00,[Elon Musk's opposition to the Global Engageme...
599,What challenges does self-managed OpenSearch d...,What are the primary technical and operational...,[Self-managed OpenSearch deployments face seve...,[- Complex setup and configuration. \n- Ongoi...,The query asks about challenges associated wit...,0.137339,0,1.00,"[- Infrastructure Management: Complex setup, r..."
600,What are the implications of Cleveland-Cliffs ...,What is the context for Cleveland-Cliffs CEO's...,[Cleveland-Cliffs CEO's plan to make another o...,[- Potential prolonged negotiations and uncert...,The query specifies key elements: the subject ...,0.179775,0,0.00,[Cleveland-Cliffs CEO plans to make another of...
601,How has the expansion of telehealth services u...,Considering the ongoing expansion of telehealt...,[The expansion of telehealth services under Me...,[- Improved access to healthcare for rural res...,The query specifies the subject matter (telehe...,0.293103,0,1.00,[- Increased access to healthcare services for...


## Evaluations

### Squad EM+F1

In [12]:
# Helper functions updated, RESTART!!
from helper_functions_qr import evaluate_squad_per_sample_multi_ref_pred

modified_set = load_dataset("json",
    data_files="./intermediate/MODIFIED_GaRAGe_UND_gpt4o_Ragas.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

qa_modified = deepcopy(modified_set)

Generating train split: 603 examples [00:00, 42532.71 examples/s]


In [13]:
squad_scored_modified, modified_f1_list, modified_em_list = evaluate_squad_per_sample_multi_ref_pred(qa_modified)
squad_scored_modified.to_json("./intermediate/MODIFIED_Gemini_GaRAGe_UND_gpt4o_new_squad.jsonl", orient="records", lines=True)

df = pd.read_json("./intermediate/MODIFIED_Gemini_GaRAGe_UND_gpt4o_new_squad.jsonl", lines=True)
df

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 47.51ba/s]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1
0,What strategies can businesses employ to mitig...,What comprehensive strategies can U.S.-based b...,"[To mitigate ASC 842 compliance challenges, bu...",[- Implement lease accounting software. \n- C...,The query seeks strategies to address ASC 842 ...,0.153846,0,0.75,[- Implement robust lease accounting software ...,0,0.316279
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Retinal Convolutional Neural...,[The Deep Retinal Convolutional Neural Network...,[- It improves accuracy by leveraging deep lea...,The query references 'Deep Retinal Convolution...,0.136546,0,0.50,[- DCNN improves speech emotion recognition by...,0,0.255319
2,"How does Zoe Law's ""Legends"" exhibition reflec...","Given that Zoë Law's ""Legends"" exhibition show...","[The ""Legends"" exhibition by Zoë Law reflects ...","['Legends' explores themes of identity, herita...",The query specifies a particular artist (Zoe L...,0.138614,0,0.25,[- It celebrates resilience by showcasing infl...,0,0.301587
3,How does the involvement of Tyco Ventures and ...,What is the impact of the $25 million investme...,[The involvement of Tyco Ventures and Integral...,[Tyco Ventures and Integral Capital Partners p...,The query identifies specific entities (Tyco V...,0.303797,0,0.50,[- Strengthened financial position for Scion P...,0,0.460177
4,How has the Drake-Kendrick Lamar feud influenc...,"Since its inception with Kendrick Lamar's ""Con...",[The feud between Drake and Kendrick Lamar has...,[- Increased focus on lyrical competition and ...,The query identifies a specific subject (the D...,0.158333,0,1.00,[- Elevated the importance of lyrical prowess ...,0,0.263158
...,...,...,...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,Considering the proposed continuing resolution...,[Elon Musk's opposition led to the president-e...,[Elon Musk's opposition to Ukraine funding rep...,The query lacks critical specificity. It refer...,0.101695,0,0.00,[Elon Musk's opposition to the Global Engageme...,0,0.128755
599,What challenges does self-managed OpenSearch d...,What are the primary technical and operational...,[Self-managed OpenSearch deployments face seve...,[- Complex setup and configuration. \n- Ongoi...,The query asks about challenges associated wit...,0.137339,0,1.00,"[- Infrastructure Management: Complex setup, r...",0,0.202429
600,What are the implications of Cleveland-Cliffs ...,What is the context for Cleveland-Cliffs CEO's...,[Cleveland-Cliffs CEO's plan to make another o...,[- Potential prolonged negotiations and uncert...,The query specifies key elements: the subject ...,0.179775,0,0.00,[Cleveland-Cliffs CEO plans to make another of...,0,0.493827
601,How has the expansion of telehealth services u...,Considering the ongoing expansion of telehealt...,[The expansion of telehealth services under Me...,[- Improved access to healthcare for rural res...,The query specifies the subject matter (telehe...,0.293103,0,1.00,[- Increased access to healthcare services for...,0,0.492537


In [14]:
modified_mean_em = np.mean(modified_em_list)  # em_scores: EM list per sample
modified_mean_f1 = np.mean(modified_f1_list)  # f1_scores F1 list per sample
print(f"New answers after modification Exact Match (avg): {modified_mean_em * 100:.2f}")
print(f"New answers after modification F1 Score (avg): {modified_mean_f1 * 100:.2f}")

original_em_list = qa_modified['original_em']
original_f1_list = qa_modified['original_f1']

original_mean_em = np.mean(original_em_list)  # em_scores: EM list per sample
original_mean_f1 = np.mean(original_f1_list)  # f1_scores F1 list per sample
print(f"Original answers Exact Match (avg): {original_mean_em * 100:.2f}")
print(f"Original answers F1 Score (avg): {original_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(modified_f1_list, original_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(modified_em_list, original_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

New answers after modification Exact Match (avg): 0.00
New answers after modification F1 Score (avg): 17.39
Original answers Exact Match (avg): 0.00
Original answers F1 Score (avg): 12.77
F1: t=7.681, p=0.0000
EM: t=nan, p=nan


### Ragas AA

In [15]:
from helper_functions_qr import answer_accuracy_modified
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

In [16]:
squad_scored_modified = load_dataset("json",
    data_files="./intermediate/MODIFIED_Gemini_GaRAGe_UND_gpt4o_new_squad.jsonl",
    split="train")
result_with_AA = await answer_accuracy_modified(squad_scored_modified, evaluator_llm)
result_with_AA.to_csv("./output_csv/MODIFIED_Gemini_GaRAGe_UND_gpt4o_all_new_scores.csv")

Generating train split: 603 examples [00:00, 41124.64 examples/s]
Calculating short answer accuracy:   3%|▎         | 16/603 [00:49<37:51,  3.87s/it]

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  8.17ba/s]


1704643

In [17]:
original_AA = list(result_with_AA["original_AA"])
modified_AA = list(result_with_AA["new_AA"])

original_mean_AA = np.mean(original_AA)
print(f"original AA (avg): {original_mean_AA * 100:.2f}")


modified_mean_AA = np.mean(modified_AA)
print(f"modified AA (avg): {modified_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(modified_AA, original_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

original AA (avg): 49.50
modified AA (avg): 56.22
AA: t=2.973, p=0.0030


## Re-Classification

In [2]:
from helper_functions_qr import (
    batch_generate_responses_qwen3,
    get_judgments_from_responses,
    run_experiment,
    prepare_test_prompts)

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


### Loading data

In [3]:
reclassify_file = "./output_csv/MODIFIED_Gemini_GaRAGe_UND_gpt4o_all_new_scores.csv"
df_reclassify_file = pd.read_csv(reclassify_file)

### Loading model

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
Qwen3_4B = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)
# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.34s/it]


cuda


### Prepare prompts

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
test_prompts = prepare_test_prompts(df_reclassify_file, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 603
Generation complete: 603 prompts
Average prompt length: 809 bytes (~202 tokens)

Analyze the following input user query:

{"query": "What comprehensive strategies can U.S.-based businesses employ to mitigate compliance challenges related to U.S. GAAP ASC 842 lease accounting, specifically addressing implementation, ongoing reporting, transitional adjustments, financial management, and operational processes?"}

Please provide your analysis in the following JSON format:

{"query": "What comprehensive strategies can U.S.-based businesses employ to mitigate compliance challenges related to U.S. GAAP ASC 842 lease accounting, specifically addressing implementation, ongoing reporting, transitional adjustments, financial management, and operational processes?", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [7]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df_reclassify_file)
test_df

100%|██████████| 121/121 [1:13:57<00:00, 36.68s/it]


,original_question,modified_question,short_answer,model_original_answer,classifier_reasoning,original_f1,original_em,original_AA,model_new_answer,new_em,new_f1,new_AA,MODIFIED_thinking,MODIFIED_model_response,MODIFIED_model_pred
0,What strategies can businesses employ to mitig...,What comprehensive strategies can U.S.-based b...,"['To mitigate ASC 842 compliance challenges, b...",['- Implement lease accounting software. \n- ...,The query seeks strategies to address ASC 842 ...,0.153846,0,0.75,['- Implement robust lease accounting software...,0,0.316279,1.00,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What comprehensive strategies...",fully specified
1,How does the Deep Retinal Convolution Neural N...,How does the Deep Retinal Convolutional Neural...,"[""The Deep Retinal Convolutional Neural Networ...",['- It improves accuracy by leveraging deep le...,The query references 'Deep Retinal Convolution...,0.136546,0,0.50,['- DCNN improves speech emotion recognition b...,0,0.255319,0.75,"<think>\nOkay, let's see. The user asked about...","{\n ""query"": ""How does the Deep Retinal Convo...",underspecified
2,"How does Zoe Law's ""Legends"" exhibition reflec...","Given that Zoë Law's ""Legends"" exhibition show...","['The ""Legends"" exhibition by Zoë Law reflects...","[""'Legends' explores themes of identity, herit...",The query specifies a particular artist (Zoe L...,0.138614,0,0.25,['- It celebrates resilience by showcasing inf...,0,0.301587,1.00,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""Given that Zoë Law's \""Legends\...",underspecified
3,How does the involvement of Tyco Ventures and ...,What is the impact of the $25 million investme...,"[""The involvement of Tyco Ventures and Integra...",['Tyco Ventures and Integral Capital Partners ...,The query identifies specific entities (Tyco V...,0.303797,0,0.50,"[""- Strengthened financial position for Scion ...",0,0.460177,1.00,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the impact of the $25 m...",underspecified
4,How has the Drake-Kendrick Lamar feud influenc...,"Since its inception with Kendrick Lamar's ""Con...",['The feud between Drake and Kendrick Lamar ha...,['- Increased focus on lyrical competition and...,The query identifies a specific subject (the D...,0.158333,0,1.00,['- Elevated the importance of lyrical prowess...,0,0.263158,1.00,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""Since its inception with Kendri...",fully specified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
598,How did Elon Musk's opposition influence the g...,Considering the proposed continuing resolution...,['Elon Musk\'s opposition led to the president...,"[""Elon Musk's opposition to Ukraine funding re...",The query lacks critical specificity. It refer...,0.101695,0,0.00,"[""Elon Musk's opposition to the Global Engagem...",0,0.128755,0.50,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""Considering the proposed cont...",underspecified
599,What challenges does self-managed OpenSearch d...,What are the primary technical and operational...,['Self-managed OpenSearch deployments face sev...,['- Complex setup and configuration. \n- Ongo...,The query asks about challenges associated wit...,0.137339,0,1.00,"['- Infrastructure Management: Complex setup, ...",0,0.202429,1.00,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""What are the primary technical ...",fully specified
600,What are the implications of Cleveland-Cliffs ...,What is the context for Cleveland-Cliffs CEO's...,"[""Cleveland-Cliffs CEO's plan to make another ...","[""- Potential prolonged negotiations and uncer...",The query specifies key elements: the subject ...,0.179775,0,0.00,"[""Cleveland-Cliffs CEO plans to make another o...",0,0.493827,0.50,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""What is the context for Clevela...",fully specified
601,How has the 

In [8]:
test_df['MODIFIED_model_pred'].value_counts(normalize=True)


MODIFIED_model_pred
fully specified    0.638474
underspecified     0.361526
Name: proportion, dtype: float64

In [9]:
test_df['MODIFIED_model_pred'].value_counts()

MODIFIED_model_pred
fully specified    385
underspecified     218
Name: count, dtype: int64

In [10]:
test_df.to_csv('./output_csv/GaRAGe_UND_Gemini_rewritten_reclassified.csv')